# Lab 13 - PyTorch and GPU

In [3]:
import torch
import torch.nn as nn
import numpy as np
import math
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'

device

'cuda'

## Ex.1) Training with Stream-based Prefetching

Goal:

Overlap **CPU data loading + H2D transfer** with **GPU compute** by:
- Using `DataLoader(pin_memory=True)` so batches arrive in pinned host memory.
- Using a **dedicated copy stream** to enqueue `to(device, non_blocking=True)`.
- Using `wait_stream` to ensure the default/compute stream waits only when needed

In [47]:
# We want to build a CUDAPrefetcher class that wraps a DataLoader by parallelizing Data Loading and GPU Compute:

class CUDAPrefetcher:
    """
    Prefetches the next batch to GPU on a dedicated CUDA stream.
    Assumes DataLoader(pin_memory=True) so H2D can be async.
    """
    def __init__(self, data_loader, device):
        self.loader = iter(data_loader)
        self.device = device
        self.copy_stream = torch.cuda.Stream(device=device)
        self.next_batch = None
        self._preload()
        
    def _to_device(self, batch):
        """Transfers batch to self.device (handles 1-deep nested structures)"""
        if isinstance(batch, (tuple, list)):
            return [t.to(self.device, non_blocking=True) for t in batch]
        return batch.to(self.device, non_blocking=True)
        
    def _preload(self):
        """Loads self.next_batch"""
        try:
            batch = next(self.loader)
            with torch.cuda.stream(self.copy_stream):
                self.next_batch = self._to_device(batch)
        except StopIteration:
            self.next_batch = None
            
    def __iter__(self):
        return self

    def __next__(self):
        if self.next_batch is None:
            raise StopIteration
    
        # Synchronize Default Stream with copy_stream!
        batch = self.next_batch
        self._preload()
        return batch

In [ ]:
# Datasets and Data Loader

transform_train = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ]
)

transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ]
)


data_root = "../../data"

train_ds = datasets.CIFAR10(
    root=data_root, train=True, download=True, transform=transform_train
)
test_ds = datasets.CIFAR10(
    root=data_root, train=False, download=True, transform=transform_test
)

train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True,
)

test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

In [10]:
image, target_class = train_ds[0]
image.shape

torch.Size([3, 32, 32])

In [49]:
class CifarCNN(nn.Module):
    
    def __init__(self, num_channels, num_classes):
        super(CifarCNN, self).__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels=num_channels, out_channels=64, kernel_size=3, stride=1, padding=1), # [64x32x32] -> [64x32x32]
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1), # [64x32x32] -> [64x32x32]
            nn.ReLU(),
            nn.MaxPool2d(2), # [64x32x32] -> [64x16x16]
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1), # [64x16x16] -> [128x16x16]
            nn.ReLU(),
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding=1), # [128x16x16] -> [128x16x16]
            nn.ReLU(),
            nn.MaxPool2d(2), # [128x16x16] -> [128x8x8]
            nn.Flatten(), # [128x8x8] -> [128 * 8 * 8]
            nn.Linear(in_features=128 * 8 * 8, out_features=256),
            nn.ReLU(),
            nn.Linear(in_features=256, out_features=num_classes)
        )
        
    def forward(self, x):
        return self.layers(x)
    
model = CifarCNN(3, 10).to(device)

In [50]:
LR = 3E-4
EPOCHS = 25

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
loss_function = nn.CrossEntropyLoss()

train_losses = list()
train_accuracies = list()
test_losses = list()
test_accuracies = list()

for epoch in range(EPOCHS):
    current_train_loss = 0.0
    train_total, train_correct = 0, 0
    
    current_test_loss = 0.0
    test_total, test_correct = 0, 0
    
    train_prefetcher = CUDAPrefetcher(train_loader, device)
    
    # Train loop
    model.train()
    for i, batch in enumerate(train_prefetcher):
        optimizer.zero_grad()
        images, target_classes = batch
        images = images.to(device)
        target_classes = target_classes.to(device)
        
        pred_logits = model(images)
        
        loss = loss_function(pred_logits, target_classes)
        
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            current_train_loss += loss.item()
            pred_classes = torch.argmax(pred_logits, dim=1)
            train_total += pred_classes.size(0)
            train_correct += (pred_classes == target_classes).sum().item()
            
    # Test loop
    model.eval()
    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            images, target_classes = batch
            images = images.to(device)
            target_classes = target_classes.to(device)
            
            pred_logits = model(images)
            loss = loss_function(pred_logits, target_classes)

            current_test_loss += loss.item()
            pred_classes = torch.argmax(pred_logits, dim=1)
            test_total += pred_classes.size(0)
            test_correct += (pred_classes == target_classes).sum().item()
        
    train_losses.append(current_train_loss)
    train_accuracies.append(train_correct / train_total)
    test_losses.append(current_test_loss)
    test_accuracies.append(test_correct / test_total)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:4d} | "
              f"Train Loss {current_train_loss:5.5f} and Accuracy {(train_correct / train_total):3.3f} | "
              f"Test Loss {current_test_loss:5.5f} and Accuracy {(test_correct / test_total):3.3f}"
              )

Epoch    0 | Train Loss 335.70048 and Accuracy 0.370 | Test Loss 55.27273 and Accuracy 0.500
Epoch   10 | Train Loss 138.17017 and Accuracy 0.752 | Test Loss 26.36528 and Accuracy 0.770
Epoch   20 | Train Loss 98.30844 and Accuracy 0.825 | Test Loss 21.75768 and Accuracy 0.814
